# NYU Depth V2 baseline (RGB-D segmentation, 40 classes)

Run the cells from top to bottom. **Before you start:** Runtime → Change runtime type → **T4 GPU**.

Everything important (prepared data, results, weights) is saved to your Google Drive in `MyDrive/rgbx/`,
so if Colab disconnects you just run the notebook again and it carries on where it stopped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/rgbx'
import os; os.makedirs(DRIVE, exist_ok=True)

## 2. Get the code
Option A (recommended): the code is on GitHub, put the repository URL below.
Option B: upload the `rgbx-baseline` folder (unzipped) into `MyDrive/rgbx/` and leave `REPO_URL` empty.

In [ ]:
REPO_URL = ''   # e.g. 'https://github.com/<your-username>/rgbx-segmentation.git'
CODE = '/content/rgbx-baseline'
if REPO_URL:
    !rm -rf {CODE} && git clone -q {REPO_URL} {CODE}
else:
    !rm -rf {CODE} && cp -r {DRIVE}/rgbx-baseline {CODE}
%cd {CODE}
!pip -q install timm
!ls

## 3. Prepare NYU Depth V2 (only slow the first time)
First time: downloads the official 2.8 GB file and converts it (about 10 to 20 minutes), then keeps a zip
of the result on Drive. Later sessions just unzip that (about 1 minute).

If the download fails (the NYU server is sometimes down), download `nyu_depth_v2_labeled.mat` yourself,
put it in `MyDrive/rgbx/`, and run the cell again.

In [ ]:
import os
DATA = '/content/data/nyuv2'
ZIP = f'{DRIVE}/nyuv2_prepared.zip'
if os.path.exists(ZIP):
    !mkdir -p /content/data && unzip -q -o {ZIP} -d /content/data
else:
    MAT = f'{DRIVE}/nyu_depth_v2_labeled.mat'
    if not os.path.exists(MAT):
        MAT = '/content/nyu_depth_v2_labeled.mat'
        !wget -c -q --show-progress -O {MAT} http://horatio.cs.nyu.edu/mit/silberman/nyu_depth_v2/nyu_depth_v2_labeled.mat
    !python prepare_nyuv2.py --mat {MAT} --out {DATA}
    !cd /content/data && zip -q -r {ZIP} nyuv2
!ls {DATA}/train/rgb | wc -l; ls {DATA}/test/rgb | wc -l   # expect 795 and 654

## 4. Look at one example (check that RGB, depth and labels line up)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from PIL import Image
i = 0
rgb = Image.open(f'{DATA}/train/rgb/{i:04d}.png'); d = np.array(Image.open(f'{DATA}/train/depth/{i:04d}.png')) / 1000
lab = np.array(Image.open(f'{DATA}/train/label/{i:04d}.png')).astype(float); lab[lab == 255] = np.nan
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].imshow(rgb); ax[0].set_title('RGB'); ax[1].imshow(d, cmap='viridis'); ax[1].set_title('depth (m)')
ax[2].imshow(lab, cmap='tab20', interpolation='nearest'); ax[2].set_title('label (40 classes, white = unlabelled)')
[a.axis('off') for a in ax]; plt.show()

## 5. Two-minute smoke test
Trains 100 steps and evaluates on 20 test images. The loss should go down from about 3.7.
This also prints the seconds per step: multiply by 10,000 to estimate the full run.

In [ ]:
!python train.py --data {DATA} --out /content/smoke --run smoke --iters 100 --eval-every 100 --max-test 20

## 6. The baseline runs
* **6a** full training set (795 images), about 1 to 2 hours on a T4.
* **6b** 5% and 10% random labelled subsets, 3 seeds each = 6 runs, roughly 20 to 40 minutes each.

Each finished run is saved in `MyDrive/rgbx/runs/` and skipped next time, so you can spread these over
several sessions. Free Colab sessions stop after a few hours; just come back and run section 1, 2, 3 and this cell again.

In [ ]:
RUNS = f'{DRIVE}/runs'
!python run_baseline.py --data {DATA} --out {RUNS} --only full

In [ ]:
!python run_baseline.py --data {DATA} --out {RUNS} --only low

## 7. (Later) RGB-only versions of the same runs
Same runs without depth. We need these for the paper (how much does depth help when labels are scarce?)
and for the complementarity score later.

In [ ]:
!python run_baseline.py --data {DATA} --out {RUNS} --modes rgb

## 8. Results table
Copy this table (or `MyDrive/rgbx/runs/summary.md` and `results.csv`) back into the project thread.

In [ ]:
!python summarize.py --csv {RUNS}/results.csv --md {RUNS}/summary.md